In [1]:
import pandas as pd
import numpy as np
from utils.neural_network import NeuralNetwork
from utils.metrics import *
import json
import time

# Architectures to test
ARCHITECTURES = [
    [30, 64, 32, 1],
    [30, 128, 64, 1],
    [30, 128, 64, 32, 1],
    [30, 256, 128, 64, 1],
    [30, 256, 128, 64, 32, 1]
]

BATCH_SIZES = [32, 64, 128]

# Fixed hyperparameters
LEARNING_RATE = 0.001
OPTIMIZER = 'adam'
EPOCHS = 200
EARLY_STOPPING_PATIENCE = 15
THRESHOLD = 0.5
RANDOM_STATE = 42

# Load pre-split datasets
df_train = pd.read_csv('dataset/creditcard_train.csv')
df_val = pd.read_csv('dataset/creditcard_val.csv')
df_test = pd.read_csv('dataset/creditcard_test.csv')

# Separate features and labels
X_train, y_train = df_train.drop('Class', axis=1), df_train['Class']
X_val, y_val = df_val.drop('Class', axis=1), df_val['Class']
X_test, y_test = df_test.drop('Class', axis=1), df_test['Class']

all_results = []

for arch in ARCHITECTURES:
    for batch_size in BATCH_SIZES:
        print(f"\nTesting: {' -> '.join(map(str, arch))} | Batch: {batch_size}")
        
        start_time = time.time()
        
        # Build and train model
        nn = NeuralNetwork(
            layers=arch,
            activation='relu',
            output_activation='sigmoid',
            random_state=RANDOM_STATE
        )
        
        nn.fit(
            X_train, y_train,
            X_val=X_val, y_val=y_val,
            epochs=EPOCHS,
            batch_size=batch_size,
            learning_rate=LEARNING_RATE,
            optimizer=OPTIMIZER,
            verbose=0,
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
        
        training_time = time.time() - start_time
        
        # Evaluate on validation set
        y_val_pred_proba = nn.predict_proba(X_val).ravel()
        y_val_pred = (y_val_pred_proba >= THRESHOLD).astype(int)
        
        # Calculate metrics
        val_pr_auc = average_precision_score(y_val, y_val_pred_proba)
        val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
        val_f1 = f1_score(y_val, y_val_pred)
        
        # Store results
        all_results.append({
            'architecture': ' -> '.join(map(str, arch)),
            'batch_size': batch_size,
            'val_pr_auc': val_pr_auc,
            'val_roc_auc': val_roc_auc,
            'val_f1': val_f1,
            'epochs_trained': len(nn.history['loss']),
            'training_time': training_time
        })
        
        print(f"Val PR-AUC: {val_pr_auc:.4f} | Val ROC-AUC: {val_roc_auc:.4f} | Time: {training_time:.1f}s")

# Create results DataFrame and sort by validation PR-AUC
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('val_pr_auc', ascending=False)

# Save results
results_df.to_csv('architecture_search_results.csv', index=False)

# Display top 5 configurations
print("\nTop 5 Configurations by Validation PR-AUC:")
print(results_df.head())

# Select best configuration
best = results_df.iloc[0]

print(f"\nOptimal Baseline Configuration:")
print(f"Architecture: {best['architecture']}")
print(f"Batch Size: {best['batch_size']}")
print(f"Validation PR-AUC: {best['val_pr_auc']:.4f}")
print(f"Validation ROC-AUC: {best['val_roc_auc']:.4f}")
print(f"Training Time: {best['training_time']:.2f}s")

# Save best configuration
best_config = {
    'architecture': eval(best['architecture'].replace(' -> ', ',')),
    'batch_size': int(best['batch_size']),
    'learning_rate': LEARNING_RATE,
    'optimizer': OPTIMIZER,
    'threshold': THRESHOLD,
    'performance': {
        'val_pr_auc': float(best['val_pr_auc']),
        'val_roc_auc': float(best['val_roc_auc']),
        'val_f1': float(best['val_f1'])
    }
}

with open('optimal_baseline_config.json', 'w') as f:
    json.dump(best_config, f, indent=2)

print("\n✓ Configuration saved to: optimal_baseline_config_simplified.json")


Testing: 30 -> 64 -> 32 -> 1 | Batch: 32
Val PR-AUC: 0.8361 | Val ROC-AUC: 0.9834 | Time: 14.0s

Testing: 30 -> 64 -> 32 -> 1 | Batch: 64
Val PR-AUC: 0.8210 | Val ROC-AUC: 0.9710 | Time: 8.4s

Testing: 30 -> 64 -> 32 -> 1 | Batch: 128
Val PR-AUC: 0.8385 | Val ROC-AUC: 0.9854 | Time: 7.7s

Testing: 30 -> 128 -> 64 -> 1 | Batch: 32
Val PR-AUC: 0.8006 | Val ROC-AUC: 0.9688 | Time: 17.9s

Testing: 30 -> 128 -> 64 -> 1 | Batch: 64
Val PR-AUC: 0.8372 | Val ROC-AUC: 0.9758 | Time: 12.5s

Testing: 30 -> 128 -> 64 -> 1 | Batch: 128
Val PR-AUC: 0.8380 | Val ROC-AUC: 0.9837 | Time: 9.0s

Testing: 30 -> 128 -> 64 -> 32 -> 1 | Batch: 32
Val PR-AUC: 0.8013 | Val ROC-AUC: 0.9750 | Time: 22.7s

Testing: 30 -> 128 -> 64 -> 32 -> 1 | Batch: 64
Val PR-AUC: 0.8479 | Val ROC-AUC: 0.9765 | Time: 16.1s

Testing: 30 -> 128 -> 64 -> 32 -> 1 | Batch: 128
Val PR-AUC: 0.8445 | Val ROC-AUC: 0.9878 | Time: 11.9s

Testing: 30 -> 256 -> 128 -> 64 -> 1 | Batch: 32
Val PR-AUC: 0.8350 | Val ROC-AUC: 0.9734 | Time: 47.4